# Week 11 — Wednesday Lab Notebook  
## Topic: Database + CRUD

**Main lab focus:** SQLite basics, CRUD operations, ORM idea, migrations concept (light), and pagination idea.

**Teacher note:**  
This lab is written in easy language for beginners. First show database concepts with simple tables. Then connect the database with Python. After that, show CRUD step by step. At the end, connect the idea with APIs so students can understand how backend and database work together.


## 3-Hour Structure

**Hour 1**
- What is a database?
- What is SQLite?
- Table, row, column, primary key
- Create a database and table
- Insert and read data

**Hour 2**
- Full CRUD using Python + SQLite
- Connect CRUD with a small Flask API
- Test endpoints inside notebook

**Hour 3**
- ORM idea using SQLAlchemy
- Migrations concept (light)
- Pagination idea
- Mini practice + lab tasks


## Setup Cell

Run the next cell once. It installs required packages only if they are missing.


In [ ]:

import sys
import subprocess
import pkgutil

required_packages = {
    "flask": "flask",
    "sqlalchemy": "SQLAlchemy"
}

for module_name, package_name in required_packages.items():
    if pkgutil.find_loader(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

print("Setup complete.")


## Part A — What is a Database?

A **database** is a place where we store data in an organized way.

Real-life examples:
- A school stores student records
- A shop stores product and sales records
- A hospital stores patient records
- A library stores book records

Without a database, data becomes difficult to search, update, and manage.

A database helps us:
- save data
- find data quickly
- update data safely
- delete data when needed


## Part B — What is SQLite?

**SQLite** is a small database system. It is very useful for beginners and small projects.

Why SQLite is good for teaching:
- easy to start
- no separate server needed
- database stored in one file
- good for small apps and practice

Example:  
If you create a file named `school.db`, that file itself becomes your database.


## Part C — Important Terms

- **Table**: a structure that stores data in rows and columns
- **Row / Record**: one complete entry
- **Column / Field**: one attribute of the data
- **Primary Key**: a unique id for each row
- **Query**: a command used to talk to the database

Example student table:

| id | name | age | city |
|---|---|---|---|
| 1 | Ali | 20 | Lahore |
| 2 | Sara | 21 | Hafizabad |


## Part D — First SQLite Database

Now we will create a very small database file and a table called `students`.


In [ ]:

import sqlite3
import os

db_name = "lab_school.db"

# remove old file for clean practice
if os.path.exists(db_name):
    os.remove(db_name)

conn = sqlite3.connect(db_name)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE students (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER,
    city TEXT
)
""")

conn.commit()
print("Database and table created successfully.")


### Teaching point

Explain to students:

- `sqlite3.connect()` opens or creates the database file
- `cursor()` is used to run SQL commands
- `CREATE TABLE` creates the structure
- `PRIMARY KEY AUTOINCREMENT` means id will increase automatically


## Part E — Insert Data

Now we will insert some rows into the table.


In [ ]:

students_data = [
    ("Ali", 20, "Lahore"),
    ("Sara", 21, "Hafizabad"),
    ("Ahmed", 19, "Faisalabad"),
    ("Ayesha", 22, "Gujranwala")
]

cursor.executemany(
    "INSERT INTO students (name, age, city) VALUES (?, ?, ?)",
    students_data
)

conn.commit()
print("Data inserted successfully.")


## Part F — Read Data

This is the **R** in CRUD.

CRUD means:
- **C** = Create
- **R** = Read
- **U** = Update
- **D** = Delete


In [ ]:

cursor.execute("SELECT * FROM students")
rows = cursor.fetchall()

for row in rows:
    print(row)


## Part G — Show Data in a Better Way with Pandas


In [ ]:

import pandas as pd

df_students = pd.read_sql_query("SELECT * FROM students", conn)
df_students


## Part H — Update Data

This is the **U** in CRUD.

Suppose we want to change Sara's city.


In [ ]:

cursor.execute(
    "UPDATE students SET city = ? WHERE name = ?",
    ("Karachi", "Sara")
)
conn.commit()

pd.read_sql_query("SELECT * FROM students", conn)


## Part I — Delete Data

This is the **D** in CRUD.

Suppose we want to remove Ahmed from the table.


In [ ]:

cursor.execute(
    "DELETE FROM students WHERE name = ?",
    ("Ahmed",)
)
conn.commit()

pd.read_sql_query("SELECT * FROM students", conn)


## Part J — Filtering Data

We can also read only selected rows.

Examples:
- all students older than 20
- students from a specific city


In [ ]:

query = "SELECT * FROM students WHERE age > 20"
pd.read_sql_query(query, conn)


In [ ]:

query = "SELECT * FROM students WHERE city = 'Karachi'"
pd.read_sql_query(query, conn)


## Part K — Why CRUD Matters in Real Apps

Every real application uses CRUD in some form.

Examples:
- student portal  
  - create student
  - read student data
  - update profile
  - delete record

- shopping app  
  - create product
  - read products
  - update price
  - delete product

- hospital system  
  - create patient record
  - read medical history
  - update report
  - delete incorrect entry


## Part L — Small Flask API Connected with SQLite

Now we connect database with backend.

We will make routes for:
- GET all students
- POST new student
- PUT update student city
- DELETE a student


In [ ]:

from flask import Flask, jsonify, request

app = Flask(__name__)

def get_connection():
    return sqlite3.connect(db_name)

@app.route("/students", methods=["GET"])
def get_students():
    conn_local = get_connection()
    conn_local.row_factory = sqlite3.Row
    cur = conn_local.cursor()
    cur.execute("SELECT * FROM students")
    rows = [dict(row) for row in cur.fetchall()]
    conn_local.close()
    return jsonify(rows)

@app.route("/students", methods=["POST"])
def add_student():
    data = request.get_json()
    name = data.get("name")
    age = data.get("age")
    city = data.get("city")

    conn_local = get_connection()
    cur = conn_local.cursor()
    cur.execute(
        "INSERT INTO students (name, age, city) VALUES (?, ?, ?)",
        (name, age, city)
    )
    conn_local.commit()
    conn_local.close()
    return jsonify({"message": "Student added successfully"}), 201

@app.route("/students/<int:student_id>", methods=["PUT"])
def update_student(student_id):
    data = request.get_json()
    new_city = data.get("city")

    conn_local = get_connection()
    cur = conn_local.cursor()
    cur.execute(
        "UPDATE students SET city = ? WHERE id = ?",
        (new_city, student_id)
    )
    conn_local.commit()
    conn_local.close()
    return jsonify({"message": "Student updated successfully"})

@app.route("/students/<int:student_id>", methods=["DELETE"])
def delete_student(student_id):
    conn_local = get_connection()
    cur = conn_local.cursor()
    cur.execute("DELETE FROM students WHERE id = ?", (student_id,))
    conn_local.commit()
    conn_local.close()
    return jsonify({"message": "Student deleted successfully"})


## Part M — Test GET Route inside Notebook


In [ ]:

with app.test_client() as client:
    response = client.get("/students")
    print("Status Code:", response.status_code)
    print("JSON:", response.json)


## Part N — Test POST Route inside Notebook


In [ ]:

new_student = {
    "name": "Bilal",
    "age": 23,
    "city": "Sialkot"
}

with app.test_client() as client:
    response = client.post("/students", json=new_student)
    print("Status Code:", response.status_code)
    print("JSON:", response.json)

pd.read_sql_query("SELECT * FROM students", conn)


## Part O — Test PUT Route inside Notebook


In [ ]:

with app.test_client() as client:
    response = client.put("/students/1", json={"city": "Islamabad"})
    print("Status Code:", response.status_code)
    print("JSON:", response.json)

pd.read_sql_query("SELECT * FROM students", conn)


## Part P — Test DELETE Route inside Notebook


In [ ]:

with app.test_client() as client:
    response = client.delete("/students/2")
    print("Status Code:", response.status_code)
    print("JSON:", response.json)

pd.read_sql_query("SELECT * FROM students", conn)


### Teaching point

At this point, make students connect the ideas:

- database stores the real data
- backend receives the request
- backend performs CRUD on the database
- backend returns response in JSON

This is how many real applications work.


## Part Q — What is an ORM?

**ORM** means **Object Relational Mapping**.

Simple idea:
- instead of writing raw SQL every time
- we can use Python classes and objects

Without ORM:
- you write SQL strings manually

With ORM:
- you create a class like `Student`
- each object behaves like a row of the table

ORM is useful because:
- code feels cleaner
- easier to manage in larger projects
- works well with Python applications


## Part R — SQLAlchemy ORM First Look

We will now create the same kind of student table using SQLAlchemy ORM.


In [ ]:

from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import declarative_base, sessionmaker

orm_db_name = "orm_school.db"
if os.path.exists(orm_db_name):
    os.remove(orm_db_name)

engine = create_engine(f"sqlite:///{orm_db_name}", echo=False)
Base = declarative_base()

class Student(Base):
    __tablename__ = "students"

    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    age = Column(Integer)
    city = Column(String)

Base.metadata.create_all(engine)

Session = sessionmaker(bind=engine)
session = Session()

print("ORM database and table created.")


## Part S — Insert Data with ORM


In [ ]:

student_objects = [
    Student(name="Hina", age=20, city="Lahore"),
    Student(name="Usman", age=24, city="Multan"),
    Student(name="Zara", age=22, city="Hafizabad")
]

session.add_all(student_objects)
session.commit()

print("ORM data inserted.")


## Part T — Read Data with ORM


In [ ]:

all_students = session.query(Student).all()

for s in all_students:
    print(s.id, s.name, s.age, s.city)


## Part U — Update Data with ORM


In [ ]:

student_to_update = session.query(Student).filter_by(name="Usman").first()
student_to_update.city = "Rawalpindi"
session.commit()

all_students = session.query(Student).all()
for s in all_students:
    print(s.id, s.name, s.age, s.city)


## Part V — Delete Data with ORM


In [ ]:

student_to_delete = session.query(Student).filter_by(name="Hina").first()
session.delete(student_to_delete)
session.commit()

all_students = session.query(Student).all()
for s in all_students:
    print(s.id, s.name, s.age, s.city)


## Part W — Raw SQL vs ORM

**Raw SQL**
- more direct
- good for learning database basics
- more control
- can become repetitive

**ORM**
- code is cleaner
- easy to connect with Python classes
- good for larger apps
- sometimes hides SQL details from beginners

Teaching advice:  
First teach raw SQL idea.  
Then show ORM as the next level.


## Part X — Migrations Concept (Light)

When your project grows, database structure can change.

Example changes:
- add a new column
- remove a column
- rename a table
- change a field type

These structure changes are called **schema changes**.

A **migration** helps track and apply these changes safely.

Simple idea:
- version 1: students table has name and age
- version 2: now we add email
- migration helps update database without rebuilding everything

In real projects:
- Flask often uses **Flask-Migrate**
- SQLAlchemy projects may use **Alembic**


## Part Y — Pagination Idea

Suppose a table has 10,000 rows.  
Should we send all rows at once?

Usually no.

Instead, we send data in smaller parts called **pages**.

Example:
- page 1 = first 5 rows
- page 2 = next 5 rows
- page 3 = next 5 rows

This makes apps faster and easier to use.


In [ ]:

# Let's create a sample table for pagination demo
cursor.execute("DROP TABLE IF EXISTS products")
cursor.execute("""
CREATE TABLE products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    price REAL
)
""")

products = [(f"Product {i}", i * 100.0) for i in range(1, 21)]
cursor.executemany("INSERT INTO products (name, price) VALUES (?, ?)", products)
conn.commit()

pd.read_sql_query("SELECT * FROM products LIMIT 5", conn)


## Part Z — Pagination with LIMIT and OFFSET


In [ ]:

page = 2
page_size = 5
offset = (page - 1) * page_size

query = f"SELECT * FROM products LIMIT {page_size} OFFSET {offset}"
pd.read_sql_query(query, conn)


## Part AA — Pagination Route Example


In [ ]:

app_pagination = Flask(__name__)

@app_pagination.route("/products", methods=["GET"])
def get_products():
    page = int(request.args.get("page", 1))
    page_size = int(request.args.get("page_size", 5))
    offset = (page - 1) * page_size

    conn_local = get_connection()
    conn_local.row_factory = sqlite3.Row
    cur = conn_local.cursor()
    cur.execute(
        "SELECT * FROM products LIMIT ? OFFSET ?",
        (page_size, offset)
    )
    rows = [dict(row) for row in cur.fetchall()]
    conn_local.close()
    return jsonify(rows)


In [ ]:

with app_pagination.test_client() as client:
    response = client.get("/products?page=3&page_size=4")
    print("Status Code:", response.status_code)
    print("JSON:", response.json)


## Part AB — Common Beginner Mistakes

1. forgetting `conn.commit()` after insert, update, or delete  
2. writing wrong SQL syntax  
3. forgetting `WHERE` in update or delete  
4. not validating user input  
5. using the same database connection carelessly  
6. sending all rows instead of paginated data  
7. confusing raw SQL and ORM syntax  
8. forgetting that API should return JSON, not Python objects directly


## Part AC — A Small Practice Scenario

**Scenario:**  
You are making a small book shop app.

Database table: `books`

Fields:
- id
- title
- author
- price

Think:
- how will Create work?
- how will Read work?
- how will Update work?
- how will Delete work?
- where can pagination help?


In [ ]:

practice_answers = {
    "Create": "Add a new book record to the books table",
    "Read": "Show all books or show one specific book",
    "Update": "Change price or author of an existing book",
    "Delete": "Remove a book record",
    "Pagination": "Useful when there are many books in the table"
}

practice_answers


## In-Class Mini Practice

Ask students to do these small tasks in class:

1. create a new table named `courses`
2. insert at least 3 rows
3. display all rows
4. update one row
5. delete one row
6. write one simple Flask route to show all courses


## Common Interview / Viva Questions from Today

1. What is a database?
2. What is SQLite?
3. What is the difference between table, row, and column?
4. What does CRUD mean?
5. Why do we use primary key?
6. What is the benefit of ORM?
7. Why are migrations useful?
8. Why do we use pagination?
9. What is the difference between raw SQL and ORM?
10. Why is `WHERE` important in update and delete?


## 10 Lab Tasks

1. Create a database file named `college.db`.  
2. Create a table `students` with columns `id`, `name`, `department`, and `semester`.  
3. Insert 5 student records into the table.  
4. Read and display all records using SQL query.  
5. Update the semester of one student.  
6. Delete one student record using id.  
7. Create a Flask route `/students` to return all students in JSON form.  
8. Create a Flask route to add one new student using POST request.  
9. Create a pagination example for a table with at least 15 rows.  
10. Write the same table once again using SQLAlchemy ORM and insert 3 records.


## Optional Homework Challenge

Build a small **Library Management Backend**.

Requirements:
- database file: `library.db`
- table: `books`
- fields: `id`, `title`, `author`, `price`
- make CRUD routes using Flask
- add one paginated GET route
- then write 5 to 7 lines explaining where ORM could help in this project


## Summary of Today

Today we learned:
- what a database is
- why SQLite is useful for beginners
- how to do CRUD using Python + SQLite
- how backend routes connect with database
- what ORM means
- why migrations matter
- why pagination is useful

This topic is very important because almost every real backend project stores data in a database.
